# Python API Example - Access Regas Slots

This guide is designed to provide an example of how to call the Access Slots API endpoint, and store the data accordingly.

__N.B. This guide is just for Access terminal slots data. If you're looking for other API data products (such as contract prices, Freight routes or Netbacks), please refer to their according code example files.__ 

### Have any questions?

If you have any questions regarding our API, or need help accessing specific datasets, please contact us at:

__data@sparkcommodities.com__

or refer to our API website for more information about this endpoint: https://www.sparkcommodities.com/api/lng-access/slot-availability.html

## 1. Importing Data

Here we define the functions that allow us to retrieve the valid credentials to access the Spark API.

This section can remain unchanged for most Spark API users.

In [ ]:
import json
import os
import sys
import pandas as pd
import numpy as np
from base64 import b64encode
from urllib.parse import urljoin
import requests
from io import StringIO
import datetime
import time

try:
    from urllib import request, parse
    from urllib.error import HTTPError
except ImportError:
    raise RuntimeError("Python 3 required")

In [ ]:
API_BASE_URL = "https://api.sparkcommodities.com"


def retrieve_credentials(file_path=None):
    """
    Find credentials either by reading the client_credentials file or reading
    environment variables
    """
    if file_path is None:
        client_id = os.getenv("SPARK_CLIENT_ID")
        client_secret = os.getenv("SPARK_CLIENT_SECRET")
        if not client_id or not client_secret:
            raise RuntimeError(
                "SPARK_CLIENT_ID and SPARK_CLIENT_SECRET environment vars required"
            )
    else:
        # Parse the file
        if not os.path.isfile(file_path):
            raise RuntimeError("The file {} doesn't exist".format(file_path))

        with open(file_path) as fp:
            lines = [l.replace("\n", "") for l in fp.readlines()]

        if lines[0] in ("clientId,clientSecret", "client_id,client_secret"):
            client_id, client_secret = lines[1].split(",")
        else:
            print("First line read: '{}'".format(lines[0]))
            raise RuntimeError(
                "The specified file {} doesn't look like to be a Spark API client "
                "credentials file".format(file_path)
            )

    print(">>>> Found credentials!")
    
    return client_id, client_secret


def do_api_post_query(uri, body, headers):
    """
    OAuth2 authentication requires a POST request with client credentials before accessing the API.
    This POST request will return an Access Token which will be used for the API GET request.
    """
    url = urljoin(API_BASE_URL, uri)

    data = json.dumps(body).encode("utf-8")

    # HTTP POST request
    req = request.Request(url, data=data, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 201. Raise an error if this is not the case
    assert response.status == 201, resp_content

    # The server returned a JSON response
    content = json.loads(resp_content)

    return content


def do_api_get_query(uri, access_token, format='json'):
    """
    After receiving an Access Token, we can request information from the API.
    Supports both JSON (default) and CSV responses.
    """
    url = urljoin(API_BASE_URL, uri)

    if format == 'json':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "application/json",
        }
    elif format == 'csv':
        headers = {
            "Authorization": "Bearer {}".format(access_token),
            "Accept": "text/csv",
        }
    else:
        raise ValueError("The format parameter only takes 'csv' or 'json' as inputs")

    print("Fetching {}".format(url))

    # HTTP GET request
    req = request.Request(url, headers=headers)
    try:
        response = request.urlopen(req)
    except HTTPError as e:
        print("HTTP Error: ", e.code)
        print(e.read())
        sys.exit(1)

    resp_content = response.read()

    # The server must return HTTP 200. Raise an error if this is not the case
    assert response.status == 200, resp_content

    # Storing response based on requested format
    if format == 'json':
        content = json.loads(resp_content)
    elif format == 'csv':
        content = resp_content

    return content


def get_access_token(client_id, client_secret):

    payload = "{}:{}".format(client_id, client_secret).encode()
    headers = {
        "Authorization": b64encode(payload).decode(),
        "Accept": "application/json",
        "Content-Type": "application/json",
    }
    body = {
        "grantType": "clientCredentials",
    }

    content = do_api_post_query(uri="/oauth/token/", body=body, headers=headers)

    print(
        ">>>> Successfully fetched an access token {}****, valid {} seconds.".format(
            content["accessToken"][:5], content["expiresIn"]
        )
    )

    return content["accessToken"]

## N.B. Credentials

Here we call the above functions, and input the file path to our credentials.

N.B. You must have downloaded your client credentials CSV file before proceeding. Please refer to the API documentation if you have not dowloaded them already.  Instructions for downloading your credentials can be found here:

https://www.sparkcommodities.com/api/request/authentication.html


In [ ]:
# Insert file path to your client credentials here
client_id, client_secret = retrieve_credentials(file_path="/tmp/client_credentials.csv")

# Authenticate:
access_token = get_access_token(client_id, client_secret)

## 2. Latest Slot Release

Here we call the latest slot release and print it in a readable format. This is done using the URL:

__/beta/terminal-slots/releases/latest/__


We then save the entire dataset as a local variable called `latest`.

In [ ]:
# Function to get the latest slot release
def get_latest_slots():
    uri = urljoin(API_BASE_URL,'/beta/terminal-slots/releases/latest/')
    headers = {
            "Authorization": "Bearer {}".format(access_token),
            "accept": "text/csv"
        }
    response = requests.get(uri, headers=headers)
    if response.status_code == 200:
        df = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(df))
    else:
        print('Bad Request')
    return df

In [ ]:
# Call latest slots function
latest = get_latest_slots()
latest.head()

## 3. Slot Release by Release Date

Here we call the slot data by choosing a specific date and print it in a readable format. This is done using the URL:

__/beta/terminal-slots/releases/{date}/__ where __date__ is the release date, in the "YYYY-MM-DD" string format.


We then save the entire dataset as a local variable called `release_df`.

In [ ]:
# Function to get the slot releases for a specific date
def get_slot_releases(date):
    uri = urljoin(API_BASE_URL, f'/beta/terminal-slots/releases/{date}/')
    headers = {
            "Authorization": "Bearer {}".format(access_token),
            "accept": "text/csv"
        }

    response = requests.get(uri, headers=headers)

    if response.status_code == 200:
        df = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(df))
        return df

    elif response.content == b'{"errors":[{"code":"object_not_found","detail":"Object not found"}]}':
        print('Bad Date')
        return None
    
    else:
        print('Bad Request')
        return None

In [ ]:
# Calling slot release function
release_df = get_slot_releases("2024-10-22")
release_df.head()

## 4. Terminal List

Here we call the list of terminals and their uuid, and print it in a readable format. This is done using the URL:

__'beta/terminal-slots/terminals/'__


We then save the entire dataset as a local variable called `terminal_list`.

In [ ]:
# Function to get the list of terminals and their uuids (as well as their start and latest release date)
def get_terminal_list():
    uri = urljoin(API_BASE_URL,'beta/terminal-slots/terminals/')
    headers = {
            "Authorization": "Bearer {}".format(access_token),
            "accept": "text/csv"
        }
    response = requests.get(uri, headers=headers)
    if response.status_code == 200:
        df = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(df))
    else:
        print('Bad Request')
    return df

In [ ]:
# Call terminal list function
terminal_list = get_terminal_list()
terminal_list.head()

## Fetching Slots Data specific to one terminal

Now that we can see all the terminal data available to us, we can start to define what terminal we want to call slots data for (by referring to `terminal_list` above).

The first step is to choose which terminal uuid (`my_uuid`), then the request will return all the historical data available for that terminal.

In [ ]:
# Function to collect and store historical slots for one specific terminal
def get_individual_terminal(terminal_uuid):
    uri = urljoin(API_BASE_URL, f'/beta/terminal-slots/terminals/{terminal_uuid}/')
    headers = {
            "Authorization": "Bearer {}".format(access_token),
            "accept": "text/csv"
        }
    response = requests.get(uri, headers=headers)
    if response.status_code == 200:
        df = response.content.decode('utf-8')
        df = pd.read_csv(StringIO(df))
        return df

    elif response.content == b'{"errors":[{"code":"object_not_found","detail":"Object not found"}]}':
        print('Bad Terminal Request')
        return None
    else:
        print('Bad Request')
        return None

In [ ]:
# Call individual terminal function (Dragon)
my_uuid = terminal_list[terminal_list['TerminalName']=='Dragon']['TerminalUUID'].tolist()[0]
dragon_hist_df = get_individual_terminal(my_uuid)
dragon_hist_df

## 5. Historical Slots

Here we collect all the historical slots by iterating over each available terminal. This is done using the URL:

__/beta/terminal-slots/terminals/{terminal_uuid}/__


In [ ]:
# Function to collect and store each terminal's historical slots data
def get_all_terminal_data(terminal_list):
    terminals_all = pd.DataFrame()
    for i in range(len(terminal_list)):
        print(terminal_list['TerminalName'].loc[i])
        terminal_df = get_individual_terminal(terminal_list['TerminalUUID'].loc[i])
        time.sleep(0.1)
        terminals_all = pd.concat([terminals_all,terminal_df])
    return terminals_all

In [ ]:
# Calling all terminal data function
all_terminal_historical = get_all_terminal_data(terminal_list)
all_terminal_historical.head()